In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
%cd "/content/drive/MyDrive/atelie-generativo-brasilia-arquitetura"

/content/drive/.shortcut-targets-by-id/1wl5jGhMjYDQlxCWqAyzFNa3sXFnfuG58/atelie-generativo-brasilia-arquitetura


In [7]:
!nvidia-smi

Thu Jul  9 23:49:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pwd
!ls

/content/drive/.shortcut-targets-by-id/1wl5jGhMjYDQlxCWqAyzFNa3sXFnfuG58/atelie-generativo-brasilia-arquitetura
dados  notebooks  README.md  relatorio


In [9]:
!find . -maxdepth 3

.
./notebooks
./notebooks/02_treino_lora.ipynb
./notebooks/01_dataset.ipynb
./README.md
./.gitignore
./.git
./.git/description
./.git/HEAD
./.git/packed-refs
./.git/config
./.git/index
./.git/COMMIT_EDITMSG
./.git/logs
./.git/logs/HEAD
./.git/logs/refs
./.git/refs
./.git/refs/heads
./.git/refs/remotes
./.git/hooks
./.git/hooks/commit-msg.sample
./.git/hooks/sendemail-validate.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/pre-commit.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/post-update.sample
./.git/hooks/applypatch-msg.sample
./.git/hooks/pre-receive.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-push.sample
./.git/hooks/update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/prepare-commit-msg.sample
./.git/info
./.git/info/exclude
./.git/objects
./.git/objects/f5
./.git/objects/f0
./.git/objects/f6
./.git/objects/ea
./.git/objects/e0
./.git/objects/pack
./.git/objects/e9
./.git/objects/e1
./.git/objects/e2

In [10]:
!pip install -q diffusers transformers accelerate peft datasets huggingface_hub safetensors

In [11]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [12]:
import os, json, shutil
from pathlib import Path
from PIL import Image, ImageOps

PROJETO = Path.cwd()
PASTA_IMAGENS = PROJETO / "dados" / "imagens"
ARQUIVO_LEGENDAS = PROJETO / "dados" / "legendas.txt"
DATASET_LORA = Path("/content/dataset_lora")

print("Projeto:", PROJETO)
print("Pasta imagens existe?", PASTA_IMAGENS.exists())
print("Legendas existe?", ARQUIVO_LEGENDAS.exists())

Projeto: /content/drive/.shortcut-targets-by-id/1wl5jGhMjYDQlxCWqAyzFNa3sXFnfuG58/atelie-generativo-brasilia-arquitetura
Pasta imagens existe? True
Legendas existe? True


In [13]:
DATASET_LORA.mkdir(parents=True, exist_ok=True)

extensoes = [".jpg", ".jpeg", ".png", ".jfif"]
imagens = sorted([p for p in PASTA_IMAGENS.iterdir() if p.suffix.lower() in extensoes])

with open(ARQUIVO_LEGENDAS, "r", encoding="utf-8") as f:
    legendas = [linha.strip() for linha in f.readlines() if linha.strip()]

print("Quantidade de imagens:", len(imagens))
print("Quantidade de legendas:", len(legendas))

if len(imagens) != len(legendas):
    print("ATENÇÃO: a quantidade de imagens e legendas está diferente.")
else:
    print("OK: quantidade de imagens e legendas está igual.")

Quantidade de imagens: 41
Quantidade de legendas: 41
OK: quantidade de imagens e legendas está igual.


In [14]:
def limpar_legenda(linha):
    linha = linha.strip()

    # Caso a linha venha no formato "nome_arquivo.jpg: legenda"
    for sep in ["|", ";", "\t", " - ", ":"]:
        if sep in linha:
            partes = linha.split(sep, 1)
            possivel_arquivo = partes[0].strip().lower()
            if possivel_arquivo.endswith((".jpg", ".jpeg", ".png", ".jfif")):
                linha = partes[1].strip()
                break

    # Garante o token do estilo no começo
    if not linha.lower().startswith("estilo_brasilia"):
        linha = "estilo_brasilia, " + linha

    return linha


metadata = []

for i, img_path in enumerate(imagens):
    legenda = limpar_legenda(legendas[i])

    novo_nome = f"id_imagem_{i+1:03d}.jpg"
    destino = DATASET_LORA / novo_nome

    img = Image.open(img_path).convert("RGB")
    img = ImageOps.fit(img, (512, 512), method=Image.Resampling.LANCZOS)
    img.save(destino, quality=95)

    metadata.append({
        "file_name": novo_nome,
        "text": legenda
    })

with open(DATASET_LORA / "metadata.jsonl", "w", encoding="utf-8") as f:
    for item in metadata:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Dataset preparado em:", DATASET_LORA)
print("Total de itens:", len(metadata))
print("Exemplo:")
print(metadata[0])

Dataset preparado em: /content/dataset_lora
Total de itens: 41
Exemplo:
{'file_name': 'id_imagem_001.jpg', 'text': 'estilo_brasilia, estilo_arquitetura_brasilia, vista panorâmica do Teatro Nacional de Brasília Claúdio Santoro exibindo toda a arquitetura do teatro em formato de pirâmide sob um céu azul limpo plano aberto'}


In [15]:
!ls /content/dataset_lora | head
!head -n 3 /content/dataset_lora/metadata.jsonl

id_imagem_001.jpg
id_imagem_002.jpg
id_imagem_003.jpg
id_imagem_004.jpg
id_imagem_005.jpg
id_imagem_006.jpg
id_imagem_007.jpg
id_imagem_008.jpg
id_imagem_009.jpg
id_imagem_010.jpg
{"file_name": "id_imagem_001.jpg", "text": "estilo_brasilia, estilo_arquitetura_brasilia, vista panorâmica do Teatro Nacional de Brasília Claúdio Santoro exibindo toda a arquitetura do teatro em formato de pirâmide sob um céu azul limpo plano aberto"}
{"file_name": "id_imagem_002.jpg", "text": "estilo_brasilia, estilo_arquitetura_brasilia, vista do topo do Teatro Nacional de Brasília Claúdio Santoro"}
{"file_name": "id_imagem_003.jpg", "text": "estilo_brasilia, estilo_arquitetura_brasilia, vista da entrada do Teatro Nacional de Brasília Claúdio Santoro mostrando a rampa de acesso e a porta do teatro"}


In [16]:
%cd /content

if not os.path.exists("/content/diffusers"):
    !git clone https://github.com/huggingface/diffusers

%cd /content/diffusers/examples/text_to_image
!ls

/content
Cloning into 'diffusers'...
remote: Enumerating objects: 126413, done.
remote: Counting objects: 100% (580/580), done.
remote: Compressing objects: 100% (296/296), done.
remote: Total 126413 (delta 416), reused 293 (delta 277), pack-reused 125833 (from 3)
Receiving objects: 100% (126413/126413), 100.27 MiB | 18.17 MiB/s, done.
Resolving deltas: 100% (94141/94141), done.
/content/diffusers/examples/text_to_image
README.md		    test_text_to_image.py
README_sdxl.md		    train_text_to_image_flax.py
requirements_flax.txt	    train_text_to_image_lora.py
requirements_sdxl.txt	    train_text_to_image_lora_sdxl.py
requirements.txt	    train_text_to_image.py
test_text_to_image_lora.py  train_text_to_image_sdxl.py


In [17]:
!accelerate config default

accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [18]:
from huggingface_hub import login
login()


In [19]:
HF_USER = "pedrohsmoura"

In [20]:
!pip uninstall -y diffusers
!pip install git+https://github.com/huggingface/diffusers

Found existing installation: diffusers 0.38.0
Uninstalling diffusers-0.38.0:
  Successfully uninstalled diffusers-0.38.0
  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-_8yrw52u
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-_8yrw52u
  Resolved https://github.com/huggingface/diffusers to commit 284419b35267dea7d4435fd0ef1eaa132bf73efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.40.0.dev0-py3-none-any.whl size=5648379 sha256=74354d75f39cd22f0824f13beb5ea9850f5b17e8d3a18d61dff133601b19f68d
  Stored in directory: /tmp/pip-ephem-wheel-cache-cyymwk3u/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers


In [21]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [22]:
import importlib.util

print("torchao instalado?", importlib.util.find_spec("torchao") is not None)

torchao instalado? False


In [23]:
!rm -rf "/content/drive/MyDrive/lora_brasilia_rank4"

In [24]:
%cd /content/diffusers/examples/text_to_image
!ls train_text_to_image_lora.py

/content/diffusers/examples/text_to_image
train_text_to_image_lora.py


In [25]:
!ls /content/dataset_lora | head
!ls /content/dataset_lora/metadata.jsonl

id_imagem_001.jpg
id_imagem_002.jpg
id_imagem_003.jpg
id_imagem_004.jpg
id_imagem_005.jpg
id_imagem_006.jpg
id_imagem_007.jpg
id_imagem_008.jpg
id_imagem_009.jpg
id_imagem_010.jpg
/content/dataset_lora/metadata.jsonl


In [26]:
from pathlib import Path
from PIL import Image, ImageOps
import json, shutil, os

PROJETO = Path("/content/drive/MyDrive/atelie-generativo-brasilia-arquitetura")

PASTA_IMAGENS = PROJETO / "dados" / "imagens"
ARQUIVO_LEGENDAS = PROJETO / "dados" / "legendas.txt"
DATASET = Path("/content/dataset")

print("Projeto existe?", PROJETO.exists())
print("Pasta de imagens existe?", PASTA_IMAGENS.exists())
print("Arquivo de legendas existe?", ARQUIVO_LEGENDAS.exists())

# Recria a pasta /content/dataset do zero
if DATASET.exists():
    shutil.rmtree(DATASET)

DATASET.mkdir(parents=True, exist_ok=True)

extensoes = [".jpg", ".jpeg", ".png", ".jfif"]
imagens = sorted([p for p in PASTA_IMAGENS.iterdir() if p.suffix.lower() in extensoes])

with open(ARQUIVO_LEGENDAS, "r", encoding="utf-8") as f:
    legendas = [linha.strip() for linha in f.readlines() if linha.strip()]

# O professor pede de 20 a 40 imagens. Você tem 41, então vamos usar as 40 primeiras.
imagens = imagens[:40]
legendas = legendas[:40]

print("Imagens usadas:", len(imagens))
print("Legendas usadas:", len(legendas))

def limpar_legenda(linha):
    linha = linha.strip()

    # Remove nome de arquivo caso a legenda esteja assim:
    # id_imagem_001.jpg: legenda...
    for sep in ["|", ";", "\t", " - ", ":"]:
        if sep in linha:
            partes = linha.split(sep, 1)
            possivel_arquivo = partes[0].strip().lower()
            if possivel_arquivo.endswith((".jpg", ".jpeg", ".png", ".jfif")):
                linha = partes[1].strip()
                break

    if not linha.lower().startswith("estilo_brasilia"):
        linha = "estilo_brasilia, " + linha

    return linha

metadata = []

for i, img_path in enumerate(imagens):
    nome_final = f"id_imagem_{i+1:03d}.jpg"
    destino = DATASET / nome_final

    img = Image.open(img_path).convert("RGB")
    img = ImageOps.fit(img, (512, 512), method=Image.Resampling.LANCZOS)
    img.save(destino, quality=95)

    metadata.append({
        "file_name": nome_final,
        "text": limpar_legenda(legendas[i])
    })

with open(DATASET / "metadata.jsonl", "w", encoding="utf-8") as f:
    for item in metadata:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Dataset criado em:", DATASET)
print("Arquivos na pasta:", len(list(DATASET.iterdir())))
print("Exemplo metadata:")
print(metadata[0])

Projeto existe? True
Pasta de imagens existe? True
Arquivo de legendas existe? True
Imagens usadas: 40
Legendas usadas: 40
Dataset criado em: /content/dataset
Arquivos na pasta: 41
Exemplo metadata:
{'file_name': 'id_imagem_001.jpg', 'text': 'estilo_brasilia, estilo_arquitetura_brasilia, vista panorâmica do Teatro Nacional de Brasília Claúdio Santoro exibindo toda a arquitetura do teatro em formato de pirâmide sob um céu azul limpo plano aberto'}


In [27]:
!find /content/dataset -maxdepth 1 -type f | head
!find /content/dataset -maxdepth 1 -iname "*.jpg" | wc -l
!head -n 3 /content/dataset/metadata.jsonl

/content/dataset/id_imagem_038.jpg
/content/dataset/id_imagem_004.jpg
/content/dataset/id_imagem_018.jpg
/content/dataset/id_imagem_036.jpg
/content/dataset/id_imagem_010.jpg
/content/dataset/id_imagem_003.jpg
/content/dataset/id_imagem_001.jpg
/content/dataset/id_imagem_029.jpg
/content/dataset/metadata.jsonl
/content/dataset/id_imagem_014.jpg
40
{"file_name": "id_imagem_001.jpg", "text": "estilo_brasilia, estilo_arquitetura_brasilia, vista panorâmica do Teatro Nacional de Brasília Claúdio Santoro exibindo toda a arquitetura do teatro em formato de pirâmide sob um céu azul limpo plano aberto"}
{"file_name": "id_imagem_002.jpg", "text": "estilo_brasilia, estilo_arquitetura_brasilia, vista do topo do Teatro Nacional de Brasília Claúdio Santoro"}
{"file_name": "id_imagem_003.jpg", "text": "estilo_brasilia, estilo_arquitetura_brasilia, vista da entrada do Teatro Nacional de Brasília Claúdio Santoro mostrando a rampa de acesso e a porta do teatro"}


In [ ]:
%cd /content/diffusers/examples/text_to_image
!ls train_text_to_image_lora.py

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_brasilia, arquitetura modernista de brasilia, concreto branco, curvas elegantes e ceu azul do cerrado" \
  --output_dir="/content/drive/MyDrive/lora_brasilia_rank8" \
  --push_to_hub \
  --hub_model_id="pedrohsmoura/lora-brasilia-rank8"

A saída de streaming foi truncada nas últimas 5000 linhas.
Loading weights: 100% 396/396 [00:00<00:00, 1502.50it/s]
Loaded safety_checker as StableDiffusionSafetyChecker from `safety_checker` subfolder of stable-diffusion-v1-5/stable-diffusion-v1-5.

Loading pipeline components...:  29% 2/7 [00:00<00:01,  3.88it/s]Instantiating AutoencoderKL model under default dtype torch.float16.
{'use_post_quant_conv', 'shift_factor', 'latents_std', 'use_quant_conv', 'scaling_factor', 'latents_mean', 'mid_block_add_attention', 'force_upcast'} was not found in config. Values will be initialized to default values.
All model checkpoint weights were used when initializing AutoencoderKL.

All the weights of AutoencoderKL were initialized from the model checkpoint at /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/vae.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Auto

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=2000 \
  --learning_rate=3e-4 \
  --lr_scheduler="cosine" \
  --rank=16 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_brasilia_futurista, arquitetura de brasilia em estilo cinematografico, luzes neon, concreto branco, ceu dramatico do cerrado, atmosfera cyberpunk brasileira" \
  --output_dir="/content/drive/MyDrive/lora-brasilia-rank4" \
  --push_to_hub \
  --hub_model_id="pedrohsmoura/lora-brasilia-rank4"